# Qwen3.5 Lab for [AIMO3](https://www.kaggle.com/competitions/ai-mathematical-olympiad-progress-prize-3)
## By [暗黑AGI](https://www.kaggle.com/boristown)
### Reference [huggingface.co](https://huggingface.co/Qwen/Qwen3.5-27B)

In [ ]:
import time
# Record the absolute start time of the entire notebook session
GLOBAL_START_TIME = time.time()
print(f"Notebook session started at: {GLOBAL_START_TIME}")

In [ ]:
!pip install -q --no-index --find-links=/kaggle/input/datasets/boristown/qwen3-5wheels/wheels/ nvidia-cudnn-cu12==9.16.0.29

!pip install -q --no-index --find-links=/kaggle/input/datasets/boristown/qwen3-5wheels/wheels/ flashinfer

!pip install -q --no-index --find-links=/kaggle/input/datasets/barnobarno/sglang-wheels sglang[all]

!pip install -q --no-index --find-links=/kaggle/input/datasets/boristown/qwen3-5wheels/wheels/ qwen-agent

In [ ]:
import os
import json
import subprocess
import tempfile
import traceback
import time
import re
import pandas as pd
import polars as pl
import kaggle_evaluation.aimo_3_inference_server

from qwen_agent.agents import Assistant
from qwen_agent.tools.base import BaseTool, register_tool

# 1. Set the maximum time limit to 4 hours and 45 minutes
TIME_LIMIT_SECONDS = 4 * 3600 + 45 * 60  

# 2. Define the Python execution tool
@register_tool('python_executor')
class PythonExecutor(BaseTool):
    description = (
        'Executes Python code locally in the current environment. '
        'Pre-installed packages include `math`, `numpy`, `sympy`, etc. '
        'STRICT LIMIT: Script must finish within 7 seconds. '
        'Use this to solve math problems, calculate equations, or process data. '
        'MUST use print() to output the final result.'
    )
    parameters = {
        "type": "object",
        "properties": {
            "code": {
                "type": "string",
                "description": "The Python script code to execute."
            }
        },
        "required": ["code"]
    }

    def call(self, params: str, **kwargs) -> str:
        try:
            code = json.loads(params).get('code', '')
        except json.JSONDecodeError:
            code = params

        with tempfile.NamedTemporaryFile(mode='w', suffix='.py', delete=False) as f:
            f.write(code)
        script_path = f.name

        try:
            result = subprocess.run(
                ["python", script_path],
                capture_output=True,
                text=True,
                timeout=7 
            )
            output = result.stdout
            
            if result.stderr:
                output += (
                    f"\n[Execution Error]:\n{result.stderr}\n"
                    "System Note: Please debug the above error. "
                    "If a purely computational approach is failing, consider solving it algebraically first. "
                    "Provide the corrected script below."
                )
            if not output.strip():
                output = "Code executed successfully, but nothing was printed. Use print() to output the answer."
            
            if len(output) > 2000:
                output = output[:1000] + "\n\n... [OUTPUT TRUNCATED] ...\n\n" + output[-1000:]
            
            return output
            
        except subprocess.TimeoutExpired:
            return (
                "System: Execution timed out after 7 seconds. "
                "The current approach is computationally too expensive. "
                "Please analyze the bottleneck internally and write a more efficient Python script (e.g., using mathematical simplification, SymPy, or dynamic programming). "
                "Please output the updated code block directly."
            )
        except Exception as e:
            error_stack = traceback.format_exc()
            return f"System Execution failed:\n{error_stack}"
        finally:
            if os.path.exists(script_path):
                os.remove(script_path)

# 3. Build the Model Class/kaggle/input/models/qwen-lm/qwen-3-5/transformers/qwen3.5-35b-a3b/1
class Model:
    """A model wrapper that manages the SGLang server and Qwen-Agent."""

    def __init__(self):
        self._model = None
        self.llm_cfg = {
            'model': 'Qwen/Qwen3.5-35B-A3B',
            'model_type': 'oai', 
            'model_server': 'http://0.0.0.0:8000/v1', 
            'api_key': 'EMPTY',
            'generate_cfg': {
                'use_raw_api': True,
                'max_tokens': 8192 * 3,
                'temperature': 1.0,
                'top_p': 1.0,
                'presence_penalty': 2.0,
                'extra_body': {
                    'chat_template_kwargs': {'enable_thinking': False},
                    'top_k': 40,
                    'min_p': 0.0,
                    'repetition_penalty': 1.0
                }
            },
        }
        
        # System prompt remains strict on the final output format
        self.system_instruction = (
            'You are an elite mathematical AI operating inside a fast-paced, multi-turn Agent framework. '
            'Your goal is to solve IMO-level problems strictly through Python code execution, not manual text derivation.\n\n'
            '# Agent Execution Rules & State Machine:\n'
            '1. CONCISE WORKFLOW: Keep text explanations extremely brief. Use inline comments within your Python code for planning.\n'
            '2. CONTINUOUS EXECUTION: In every intermediate round, instantly invoke the `python_executor` tool to test hypotheses, calculate, or find patterns.\n'
            '3. READ & REACT: Based on the observation, write the next block of code to debug or progress. Keep text outside code blocks to an absolute minimum.\n'
            '4. HARD LIMITS: Python execution is strictly capped at 7 seconds. Optimize algorithms to avoid timeouts.\n'
            '5. MANDATORY VERIFICATION PHASE (CRITICAL): You are FORBIDDEN from outputting a final answer immediately after finding a potential solution. '
            'Before your final output, you MUST write a dedicated verification script. This script must verify the proposed answer using a COMPLETELY DIFFERENT mathematical method, or by plugging the answer back into the original problem constraints. '
            'Only if this independent verification script runs successfully and confirms the exact same result, may you proceed to the final output.\n'
            '6. ZERO GUESSING: If the verification script fails, times out, or yields a contradiction, you MUST discard the answer and write new code to find the flaw.\n\n'
            '# FINAL OUTPUT FORMAT (CRITICAL):\n'
            'The final answer must be a single non-negative integer between 0 and 99999.\n'
            'Once you have explicitly received a positive confirmation from your verification script, STOP CODING.\n'
            'Your final response MUST be exclusively the boxed answer and absolutely nothing else. '
            'DO NOT write summaries, reasoning, or text explanations.\n'
            'Example: \\boxed{42}'
        )
    def load(self):
        """Start the SGLang server and wait until ready."""
        print("Loading model and starting SGLang server...")
        import requests
        
        # SGLang JIT Workaround for Kaggle read-only environment
        custom_lib_dir = "/tmp/custom_cuda_lib"
        os.makedirs(custom_lib_dir, exist_ok=True)
        libcuda_path = os.popen("find /usr -name libcuda.so.1 2>/dev/null | head -n 1").read().strip()
        if libcuda_path:
            os.system(f"ln -sf {libcuda_path} {custom_lib_dir}/libcuda.so")
            os.environ["LIBRARY_PATH"] = f"{custom_lib_dir}:{os.environ.get('LIBRARY_PATH', '')}"
            os.environ["LD_LIBRARY_PATH"] = f"{custom_lib_dir}:{os.environ.get('LD_LIBRARY_PATH', '')}"
        os.environ["SGLANG_DISABLE_CUDNN_CHECK"] = "1"
        
        command = [
            "python", "-m", "sglang.launch_server",
            "--model-path", "/kaggle/input/models/qwen-lm/qwen-3-5/transformers/qwen3.5-35b-a3b/1",
            "--port", "8000",
            "--tp-size", "1",                      
            "--mem-fraction-static", "0.85",
            "--context-length", "131072",
            "--max-prefill-tokens", "8192",
            #"--reasoning-parser", "qwen3",
            "--tool-call-parser", "qwen3_coder"
        ]
        
        self.process = subprocess.Popen(command, stdout=open('server.log', 'w'), stderr=subprocess.STDOUT)
        
        api_url = "http://0.0.0.0:8000/v1/models"
        for attempt in range(1, 30):
            if self.process.poll() is not None:
                print("❌ SGLang process crashed! Check server.log.")
                break
            try:
                response = requests.get(api_url, timeout=5)
                if response.status_code == 200:
                    print(f"✅ Server started successfully!")
                    break
            except:
                pass
            time.sleep(60)

        def _predict(problem_text: str) -> int:
            try:
                # Check global time limit using GLOBAL_START_TIME from Cell 1
                elapsed = time.time() - GLOBAL_START_TIME
                if elapsed >= TIME_LIMIT_SECONDS:
                    print(f"⚠️ Global time limit reached ({elapsed:.0f}s elapsed). Outputting 0 directly.")
                    return 0
            except NameError:
                pass # Defensive programming: if Cell 1 wasn't run, avoid crashing
                
            bot = Assistant(llm=self.llm_cfg, function_list=['python_executor'], system_message=self.system_instruction)
            messages = [{'role': 'user', 'content': problem_text}]
            final_answer = 0
            
            # [UPDATED]: Setup variables for per-problem timeout
            PROBLEM_TIME_LIMIT = 900
            problem_start_time = time.time()
            
            try:
                responses_gen = bot.run(messages=messages)
                prev_len = len(messages)
                round_start_time = time.time()
                responses_list = []
                
                for responses in responses_gen:
                    responses_list = responses
                    
                    # [UPDATED]: Circuit Breaker - Check single problem timeout
                    if time.time() - problem_start_time > PROBLEM_TIME_LIMIT:
                        print(f"\n⚠️ [Circuit Breaker] Time limit exceeded for this problem ({PROBLEM_TIME_LIMIT}s). Halting reasoning loop!")
                        break # Break the generator loop immediately
                        
                    if len(responses) > prev_len:
                        msg_to_print = responses[prev_len - 1]
                        role = msg_to_print.get('role', 'unknown')
                        content = msg_to_print.get('content', '') or ''
                        if 'function_call' in msg_to_print:
                            content += str(msg_to_print['function_call'])
                            
                        round_elapsed = time.time() - round_start_time
                        speed = (len(content) / 4.0) / round_elapsed if round_elapsed > 0 else 0
                        
                        if role in ('tool', 'observation') and any(x in content for x in ['Error', 'Exception', 'Traceback']):
                            truncated_content = content[:1500] + ('...\n[TRUNCATED]' if len(content) > 1500 else '')
                            print(f"Round {prev_len} | {role} | Speed: {speed:.1f} t/s\n[ERROR LOG]:\n{truncated_content}")
                        else:
                            snippet = content.replace('\n', ' ')[:50]
                            print(f"Round {prev_len} | {role} | {snippet}... | Total chars: {len(content)} | Speed: {speed:.1f} t/s")
                        
                        prev_len = len(responses)
                        round_start_time = time.time()
                
                if len(responses_list) > 0:
                    last_msg = responses_list[-1]
                    content = last_msg.get('content', '') or ''
                    print(f"Round {len(responses_list)} | {last_msg.get('role')} | [FINAL CONTENT]:\n{content}")
                    
                    matches = re.findall(r'\\boxed\{(\d+)\}', content)
                    if matches:
                        try:
                            ans = int(matches[-1])
                            final_answer = ans % 100000 if ans > 99999 or ans < 0 else ans
                        except:
                            pass
                            
            except Exception as e:
                print(f"Error during reasoning: {e}")
                
            return final_answer

        return _predict

    def predict(self, problem: str):
        if self._model is None:
            self._model = self.load()
        return self._model(problem)

model = Model()


# =====================================================================
# 4. Local Evaluation Monitor (Side-car logic: safely ignored during submission)
# =====================================================================
IS_LOCAL_RUN = not os.getenv('KAGGLE_IS_COMPETITION_RERUN')
expected_answers = {}
local_correct = 0
local_total = 0

if IS_LOCAL_RUN:
    try:
        # Silently load the reference dictionary ONLY when running locally
        ref_path = '/kaggle/input/ai-mathematical-olympiad-progress-prize-3/reference.csv'
        if os.path.exists(ref_path):
            df_ref = pd.read_csv(ref_path)
            if 'answer' in df_ref.columns and 'id' in df_ref.columns:
                expected_answers = dict(zip(df_ref['id'], df_ref['answer']))
    except Exception as e:
        pass


def predict(id_: pl.Series, problem: pl.Series) -> pl.DataFrame:
    """Make a prediction for the AIMO3 server."""
    global local_correct, local_total
    
    id_val = id_.item(0)
    problem_text = problem.item(0)
    
    print(f"\n========== Evaluating ID: {id_val} ==========")
    prediction = model.predict(problem_text)
    print(f"Result for {id_val}: {prediction}")
    
    if IS_LOCAL_RUN and id_val in expected_answers:
        expected = expected_answers[id_val]
        is_correct = str(prediction) == str(expected)
        
        if is_correct:
            local_correct += 1
            print(f"✅ Auto-Eval: Correct! (Expected: {expected})")
        else:
            print(f"❌ Auto-Eval: Incorrect! (Expected: {expected})")
            
        local_total += 1
        current_acc = (local_correct / local_total) * 100
        print(f"📊 Running Accuracy: {local_correct}/{local_total} ({current_acc:.2f}%)")
        
    print("=============================================\n")
    
    return pl.DataFrame({'id': id_val, 'answer': prediction})

# =====================================================================

inference_server = kaggle_evaluation.aimo_3_inference_server.AIMO3InferenceServer(predict)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
else:
    original_csv = '/kaggle/input/ai-mathematical-olympiad-progress-prize-3/reference.csv'
    mock_csv = '/kaggle/working/mock_reference.csv'
    
    if os.path.exists(original_csv):
        df_clean = pd.read_csv(original_csv)
        if 'answer' in df_clean.columns:
            df_clean = df_clean.drop(columns=['answer'])
        df_clean.to_csv(mock_csv, index=False)
        print("📁 [Local Setup] Created mock test set without 'answer' column.")
        
    inference_server.run_local_gateway((mock_csv,))